In [1]:
import requests
import json

BASE_URL = "http://127.0.0.1:8000"

# Test 1: Health check
response = requests.get(f"{BASE_URL}/")
print("Health check:", response.json())

Health check: {'status': 'API is running', 'model': 'XGBoost', 'features': 20}


In [2]:
# Test 2: Single prediction
payload = {
    "store_nbr": 1,
    "family": "GROCERY I",
    "date": "2017-08-16",
    "onpromotion": 5,
    "oil_price": 48.5,
    "is_national_holiday": 0,
    "transactions": 2500.0,
    "sales_lag_7": 2800.0,
    "sales_lag_14": 2750.0,
    "sales_lag_28": 2700.0,
    "sales_rolling_7day_avg": 2780.0,
    "sales_rolling_28day_avg": 2720.0
}

response = requests.post(f"{BASE_URL}/predict", json=payload)

print(f"Status code: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

Status code: 200
Response: {
  "store_nbr": 1,
  "family": "GROCERY I",
  "date": "2017-08-16",
  "store_type": "D",
  "predicted_sales": 2987.47
}


In [3]:
# Test 3: Check error handling (invalid family)
bad_payload = payload.copy()
bad_payload['family'] = "FAKE CATEGORY"

response = requests.post(f"{BASE_URL}/predict", json=bad_payload)
print(f"Status code: {response.status_code}")
print(f"Error response: {response.json()}")

Status code: 400
Error response: {'detail': "Unknown family. Valid options: ['AUTOMOTIVE', 'BABY CARE', 'BEAUTY', 'BEVERAGES', 'BOOKS', 'BREAD/BAKERY', 'CELEBRATION', 'CLEANING', 'DAIRY', 'DELI', 'EGGS', 'FROZEN FOODS', 'GROCERY I', 'GROCERY II', 'HARDWARE', 'HOME AND KITCHEN I', 'HOME AND KITCHEN II', 'HOME APPLIANCES', 'HOME CARE', 'LADIESWEAR', 'LAWN AND GARDEN', 'LINGERIE', 'LIQUOR,WINE,BEER', 'MAGAZINES', 'MEATS', 'PERSONAL CARE', 'PET SUPPLIES', 'PLAYERS AND ELECTRONICS', 'POULTRY', 'PREPARED FOODS', 'PRODUCE', 'SCHOOL AND OFFICE SUPPLIES', 'SEAFOOD']"}


In [4]:
# Test 4: Batch prediction (multiple stores at once)
batch_payload = {
    "predictions": [
        {**payload, "store_nbr": 1, "family": "GROCERY I"},
        {**payload, "store_nbr": 2, "family": "BEVERAGES"},
        {**payload, "store_nbr": 3, "family": "PRODUCE"},
    ]
}

response = requests.post(f"{BASE_URL}/predict/batch", json=batch_payload)
print(f"Batch predictions: {response.json()['count']} results")
for pred in response.json()['predictions']:
    print(f"  Store {pred['store_nbr']} | {pred['family']} → {pred['predicted_sales']}")

Batch predictions: 3 results
  Store 1 | GROCERY I → 2987.47
  Store 2 | BEVERAGES → 2757.4
  Store 3 | PRODUCE → 2539.49
